# Student Engagement Medallion Pipeline — Complete Build Story

## Executive Summary

This pipeline transforms raw PostgreSQL CRM and engagement data into a **Customer Health Profile** that scores 1,584 coaching clients across three engagement dimensions — Slack communication, platform login, and meeting attendance — producing a single health score (0–100) and band (GOOD / AVERAGE / POOR / NO DATA) for each client.

### What was built

| Layer | Tables | Rows (total) | Purpose |
|---|---|---|---|
| Bronze | 9 | 17,246,500 | Raw JSON landing from PostgreSQL via JDBC |
| Silver | 11 | 44,223,263 | Parsed, flattened, deduplicated, struct-exploded |
| Gold | 14 | 853,589 | 7 engagement metrics, 2 lookup tables, health profile |

### What the pipeline produces today

| Metric | Value |
|---|---|
| Coaching clients scored | 969 of 1,584 (61.2%) |
| GOOD (score ≥75) | 190 clients (12.0%) |
| AVERAGE (score 50–74) | 303 clients (19.1%) |
| POOR (score <50) | 476 clients (30.1%) |
| NO DATA (unscoreable) | 615 clients (38.8%) |
| Metric coverage: Calendly meetings | 1,240 clients (78.3%) |
| Metric coverage: Sentiment | 1,378 clients (87.0%) |
| Metric coverage: CRM meetings | 940 clients (59.3%) |
| Metric coverage: Email recency | 826 clients (52.1%) |
| Metric coverage: Platform login | 356 clients (22.5%) |

### Key engineering improvement this session

The Moodle platform-login join was refactored from a single-column match (`USERNAME`, 23 clients) to a multi-column UNION match (`USERNAME` ∪ `FIRSTNAME` ∪ `LASTNAME`, 356 clients) — a **15× coverage increase** that moved 333 clients from unscoreable ("NO DATA") to scored health bands.

---
## 1. Architecture Overview

The pipeline follows Databricks Medallion Architecture with three layers, orchestrated by a Databricks Job (ID: 299306717398586) running 17 tasks daily at 8:00 AM EST on serverless compute.

```
PostgreSQL (raw schema, 9 tables)
    │
    ▼  JDBC — daily append (WHERE DATE(INSERT_DATE) = CURRENT_DATE)
BRONZE — crm_ingestion.bronze (9 Delta tables, ~17.2M rows)
    │
    ▼  %run utils → process_table() → parse_bronze_table → flatten_structs → write_silver
SILVER — crm_ingestion.silver (11 tables, ~44.2M rows)
    │
    ▼  5 Gold notebooks: metric tables + lookups → flags → patterns → scores → bands → MERGE SCD1
GOLD — crm_ingestion.gold (14 tables, customer_health_profile)
```

**Execution order:**
1. **Bronze** (1 task): `1_2_Bronze_ingestion` — loops 9 PostgreSQL tables, append-only
2. **Silver** (11 tasks, parallel): Each notebook calls `%run ../utils` then `process_table()`
3. **Gold** (5 tasks, dependency chain): 3_1 → 3_2 → 3_3 → 3_5, with 3_4 parallel to 3_3

### Folder Structure
```
student-engagement/
├── 00_README_.md
├── utils                                  ← Shared engine (10 cells, 7 functions)
├── 1_Bronze/
│   ├── 1_1_Bronze_config                  ← JDBC connection settings
│   └── 1_2_Bronze_ingestion               ← Loop over 9 source tables
├── 2_Silver/                              ← 11 processing notebooks
│   ├── 2_1_01_Silver_leads_processed
│   ├── 2_1_02_Silver_leads_processed_custom
│   ├── 2_1_03_01_Silver_leads_processed_contacts
│   ├── 2_1_03_02_Silver_leads_processed_contacts_emails
│   ├── 2_2_01_Silver_lead_activites_processed
│   ├── 2_2_02_Silver_lead_activites_email_body
│   ├── 2_3_Silver_close_crm_users_processed
│   ├── 2_4_Silver_custom_activites_processed
│   ├── 2_5_Silver_calendly_scheduled_events_processed
│   ├── 2_6_Silver_student_sentiment_processed
│   └── 2_7_Silver_mdl_users_processed
└── 3_Gold/
    ├── 3_1_Gold_Leads_Activities_Summary
    ├── 3_2_Gold_Sales_Details
    ├── 3_3_Gold_Metrics                   ← 7 engagement metric tables
    ├── 3_4_Gold_Lookup_Tables             ← lead_info + lead_emails
    └── 3_5_Gold_Customer_Health_Profile    ← Scoring + MERGE SCD Type 1
```

---
## 2. Bronze Layer — Raw Landing Zone (9 tables, 17.2M rows)

Bronze ingestion uses a single notebook (`1_2_Bronze_ingestion`) that loops over 9 PostgreSQL source tables via JDBC. Each run appends only current-day records (`WHERE DATE(INSERT_DATE) = CURRENT_DATE`). No CDC, no MERGE — simple daily append.

All bronze tables share 3 columns: `raw_data` (JSON string), `insert_date` (timestamp), `bronze_updated_at` (timestamp).

| # | Bronze Table | Source | Rows | Used Downstream |
|---|---|---|---|:---:|
| 1 | `leads_raw` | `raw.leads_raw` | 438,798 | ✅ |
| 2 | `lead_activites_raw` | `raw.lead_activites_raw` | 362,767 | ✅ |
| 3 | `close_crm_users_raw` | `raw.close_crm_users_raw` | 78,632 | ✅ |
| 4 | `custom_activites_raw` | `raw.custom_activites_raw` | 140 | ✅ |
| 5 | `all_payments` | `raw.all_payments` | 236,393 | ❌ |
| 6 | `calendly_scheduled_events` | `raw.calendly_scheduled_events` | 13,454,841 | ✅ |
| 7 | `student_sentiment` | `raw.student_sentiment` | 184,716 | ✅ |
| 8 | `mdl_users_raw` | `raw.mdl_users_raw` | 1,967,563 | ✅ |
| 9 | `lead_merges` | `raw.lead_merges` | 277,435 | ❌ |

Tables 5 (`all_payments`) and 9 (`lead_merges`) are loaded to bronze but not processed into silver — they have no downstream consumer in the current pipeline.

**Why bronze is append-only:** The PostgreSQL source exports full snapshots daily. Bronze preserves every daily snapshot, allowing point-in-time reconstruction. Silver handles deduplication.

---
## 3. Utils Engine — Shared Silver Processing (10 cells, 7 functions)

**Notebook:** `utils` (ID: 1455530248234575)  
**Path:** `/Users/scottsbv@gmail.com/student-engagement/utils`  
**Called by:** Every silver notebook via `%run /Users/scottsbv@gmail.com/student-engagement/utils`

The utils notebook is the engine that powers all 11 silver processing notebooks. Each function is described once here; subsequent sections reference them by name.

### Cell 2 — Configuration

Sets catalog (`crm_ingestion`), schemas (`bronze`, `silver`), and imports from `pyspark.sql.functions` and `pyspark.sql.types`. Defines `_PROBLEM_KEYS`, `_TEMP_TABLES`, `_CDC_WATERMARK`, and `PIPELINE_NUMBERS` globals.

### Cell 3 — `repair_json_udf`

A Spark UDF (`udf(StringType())(_parse_activity_final)`) that repairs malformed Close CRM JSON. This is the most complex function in the engine — it handles 5 distinct JSON pathologies:

1. **JSON_OBJECT wrapper**: Close CRM wraps payloads in `{"JSON_OBJECT": "..."}`. The UDF extracts and parses the inner string.
2. **"data" key unwrapping**: Some tables wrap records in `{"data": [...]}`. The UDF returns just the array.
3. **Single-quote delimiters**: PostgreSQL exports use single quotes instead of double quotes. The UDF replaces `'` → `"` for delimiters while preserving apostrophes in words (e.g., `Prospect's Name` → `ProspectAPOSTROPHEs Name` → `Prospect's Name`).
4. **Problem key nulling**: Free-text fields (`description`, `notes`, `body_text`, `subject`, `text`, `note`) contain single quotes and special characters that break JSON parsing. The UDF nulls these values via scan-based string matching before quote replacement, and again programmatically as a safety net on parsed dicts.
5. **Array detection**: Tracks whether the UDF returns a JSON array (each row contains multiple records) vs a single object, so `parse_bronze_table` can explode correctly.

**Used by:** `parse_bronze_table()` (cell 4) — runs once during temp table materialization, then all subsequent reads are UDF-free.

### Cell 4 — `parse_bronze_table(bronze_table, watermark=None, limit=None)`

Reads a bronze table, repairs JSON via the UDF, discovers schema, and returns a flattened DataFrame. Steps:

1. Reads `crm_ingestion.bronze.{bronze_table}`, optionally filtered by CDC watermark
2. Detects if UDF repair is needed (checks for `JSON_OBJECT` wrapper or single-quote delimiters on a sample row)
3. If repair needed: runs `repair_json_udf` → materializes to a temp table (UDF runs ONCE) → reads from temp table (all reads are fast, no UDF)
4. If clean JSON: skips UDF, uses `raw_data` directly
5. Detects if UDF returns arrays (each row is a JSON array of records) vs single objects
6. Schema discovery via `schema_of_json_agg()` on up to 200 sampled rows — infers the DDL schema string
7. Deduplicates field names (Close CRM has duplicate keys in some tables)
8. Parses with `from_json()` using `ArrayType<StructType>` if UDF returns arrays, or `StructType` if single objects
9. Explodes top-level array into individual rows

Returns: a flat DataFrame with `bronze_insert_date`, `bronze_updated_at`, plus all discovered fields.

**Critical fix:** When the UDF returns a JSON array string (e.g., `[{...}, {...}]`), `from_json()` with a `StructType` schema produces all-NULL fields. The fix tracks `_udf_returns_array` flag during sample parsing and uses `ArrayType(deduped_st)` for `from_json()` when true. This fixed `close_crm_users_raw` (18.3M rows, was all-NULL) and `lead_activites_raw` (9.2M rows, was all-NULL).

### Cell 5 — `categorize_fields(df)` and `flatten_structs(df, skip=None)`

- **`categorize_fields(df)`**: Inspects DataFrame schema and returns three lists: `(scalars, arrays, structs)`. Used by `process_table()` to decide which columns to explode vs flatten.
- **`flatten_structs(df, skip=None)`**: Flattens `StructType` columns into individual columns (e.g., `envelope.bcc` → `envelope_bcc`). Skips the `custom` struct for separate extraction by `extract_custom()`. Default is `skip_custom=True`.

### Cell 7 — `write_silver(df, table_name)`

Writes a DataFrame to `crm_ingestion.silver.{table_name}` as a Delta table.

- **CDC mode** (when `_CDC_WATERMARK` is set): `DELETE WHERE bronze_updated_at > watermark` then `APPEND` new rows. This is a delete+insert pattern.
- **Full load mode** (no watermark): `OVERWRITE` with `overwriteSchema: true`. This is the default for first runs and full reprocessing.
- Uses Delta `columnMapping.mode = name`, `minReaderVersion = 2`, `minWriterVersion = 5`.

**Note:** `add_record_hash()` was previously called here to add an MD5 all-columns fingerprint. This was removed because it had no functional use — the hash cannot be used for joins, identity resolution, or CDC (which is not implemented). New silver writes no longer add `record_hash`.

### Cell 8 — `explode_to_child(df, parent_name, array_col, carry_cols, depth=0, num_path="", parent_pk=None, dry_run=False, recurse=True)`

Explodes an array column into a child table. For example, `leads_processed.contacts` (array of contact structs) becomes `leads_processed_contacts` (one row per contact).

- Carries parent FK columns (`bronze_insert_date`, `bronze_updated_at`, parent PK)
- Renames parent PK to `{parent_name}_id` for FK clarity (always, not just on collision)
- Recurses on nested arrays within the exploded struct (e.g., contacts → emails → a grandchild table)
- `dry_run=True`: prints the discovered schema without writing the table

**Used by:** `process_table()` — called for each array column to discover and print child table lineage.

### Cell 9 — `extract_custom(df, parent_name, carry_cols, num_path, parent_pk=None, dry_run=False)`

Extracts the `custom` struct and `cf_` prefixed fields into a separate `_custom` child table. Builds a `custom_fields` MAP column on the parent table for downstream access.

The `custom_fields` MAP is critical for the Gold layer — it holds CRM custom fields like `CSM`, `Tier`, `Avatar`, `Contracted_Value`, `DAYS_SINCE_LAST_EMAIL`, `SENTIMENTS_LAST_30_DAYS`, etc. These pre-calculated values from Close CRM are used as a validation reference against the pipeline's self-calculated metrics.

**Used by:** `process_table()` — called before array explosion to extract custom fields.

### Cell 10 — `process_table(bronze_name, silver_name=None, only_arrays=None, flatten=False)`

The orchestrator. This is the single entry point called by all 11 silver notebooks:

```python
process_table('leads_raw', 'leads_processed', only_arrays=['contacts'], flatten=False)
```

Execution flow:
1. `get_silver_watermark()` → check for CDC mode (returns `MAX(bronze_updated_at)` or None)
2. `parse_bronze_table()` → repair JSON, discover schema, parse, explode arrays (see above)
3. `flatten_structs()` → flatten struct columns if `flatten=True` (default: False)
4. `find_pk()` → identify primary key column for FK relationships
5. `extract_custom()` → extract custom fields into `_custom` table + `custom_fields` MAP (dry_run)
6. `write_silver()` → write parent table to Delta
7. `categorize_fields()` → identify array columns
8. `explode_to_child()` → for each array, discover and print child table schema (dry_run)
9. `_discover_lineage_tree()` → auto-discover hierarchical lineage from existing tables
10. `print_lineage_tree()` → print numbered tree
11. Clean up temp tables, reset CDC watermark

**`only_arrays` parameter:** Controls which array columns to explode. `None` = all arrays, `[]` = none, `['contacts']` = only contacts. Most silver notebooks use `only_arrays=[]` (parent table only) except `leads_raw` (only_arrays=['contacts']) and `lead_activites_raw` (only_arrays=['attendees']).

**`flatten` parameter:** Default is `False` — struct columns (e.g., `envelope`, `summary`) are preserved as struct types. Setting to `True` expands them into individual columns.

---
## 4. Silver Layer — Cleaned & Normalized (11 tables, 44.2M rows)

Each silver notebook calls `process_table()` from the utils engine. The table below shows which utils functions are active in each notebook and the resulting row counts.

| # | Notebook | Bronze Source | Silver Table | Rows | Utils Functions Used |
|---|---|---|---|---|---|
| 1 | `2_1_01_Silver_leads_processed` | `leads_raw` | `leads_processed` | 381,328 | `parse_bronze_table` → `flatten_structs` → `extract_custom` → `write_silver` → `explode_to_child(contacts)` |
| 2 | `2_1_02_Silver_leads_processed_custom` | `leads_raw` | `leads_processed_custom` | 381,328 | `parse_bronze_table` → `extract_custom` (standalone) |
| 3 | `2_1_03_01_Silver_leads_processed_contacts` | `leads_raw` | `leads_processed_contacts` | 466,420 | `parse_bronze_table` → `explode_to_child(contacts)` |
| 4 | `2_1_03_02_Silver_leads_processed_contacts_emails` | `leads_raw` | `leads_processed_contacts_emails` | 564,300 | `parse_bronze_table` → `explode_to_child(contacts)` → `explode_to_child(emails)` |
| 5 | `2_2_01_Silver_lead_activites_processed` | `lead_activites_raw` | `lead_activites_processed` | 9,234,618 | `parse_bronze_table` → `write_silver` |
| 6 | `2_2_02_Silver_lead_activites_email_body` | `lead_activites_raw` | `lead_activites_email_body` | 1,815,569 | `parse_bronze_table` (standalone extract of `body_text_quoted[0].text`) |
| 7 | `2_3_Silver_close_crm_users_processed` | `close_crm_users_raw` | `close_crm_users_processed` | 18,327,493 | `parse_bronze_table` → `write_silver` |
| 8 | `2_4_Silver_custom_activites_processed` | `custom_activites_raw` | `custom_activites_processed` | 3,530 | `parse_bronze_table` → `write_silver` |
| 9 | `2_5_Silver_calendly_scheduled_events_processed` | `calendly_scheduled_events` | `calendly_scheduled_events_processed` | 12,213,327 | `parse_bronze_table` → `write_silver` |
| 10 | `2_6_Silver_student_sentiment_processed` | `student_sentiment` | `student_sentiment_processed` | 157,884 | `parse_bronze_table` → `write_silver` |
| 11 | `2_7_Silver_mdl_users_processed` | `mdl_users_raw` | `mdl_users_processed` | 1,653,776 | `parse_bronze_table` → `write_silver` |

### Why `lead_activites_email_body` exists separately

`parse_bronze_table` discovers schema from 200 sampled rows via `schema_of_json_agg()`. If those rows don't include all activity types, type-specific fields are missing. `lead_activites_processed` captures 76 of 135 fields — the missing 59 are email-specific fields including `body_text_quoted`, which contains the email body text.

The `lead_activites_email_body` notebook re-parses the bronze table using `parse_bronze_table` (which discovers the full 135-field schema on a different sample) and extracts `body_text_quoted[0].text` as `body_text`. This field contains Calendly's "A new event has been scheduled." message — the signal used to detect CRM-recorded meetings.

### Key silver tables for the Gold layer

**`leads_processed`** (381,328 rows): The lead dimension. Key columns: `id` (PK), `status_label` (filters to 'Coaching Client'), `date_updated`, `contacts` (array of contact structs with `name`, `email`, `phone`).

**`leads_processed_custom`** (381,328 rows): CRM pre-calculated custom fields extracted by `extract_custom()`. Key columns: `leads_processed_id` (FK), `CSM`, `Tier`, `Avatar`, `Contracted_Value`, `DAYS_SINCE_LAST_EMAIL`, `DAYS_SINCE_LAST_LOGIN`, `SENTIMENTS_LAST_30_DAYS`, `FINAL_SCORE`, `HEALTH_BAND`. These are Close CRM's own values — the Gold layer builds self-calculated versions from raw activity data to validate and replace them.

**`leads_processed_contacts`** (466,420 rows): Exploded from `leads_processed.contacts` via `explode_to_child()`. Key columns: `leads_processed_id` (FK), `name` (the `user_xxx` hash — see §6 Cross-System Identity Resolution), `email`, `phone`. This is the bridge table for all cross-system joins.

**`lead_activites_processed`** (9.2M rows): All CRM activities (Email, Call, Meeting, Note, SMS, CustomActivity). Key columns: `id`, `lead_id`, `type`, `status`, `activity_at`, `direction`, `disposition`, `note`, `text` (SMS only, NOT email body).

**`lead_activites_email_body`** (1.8M rows): Email body text extracted from `body_text_quoted[0].text`. Key columns: `activity_id` (FK to `lead_activites_processed.id`), `lead_id`, `body_text`.

**`calendly_scheduled_events_processed`** (12.2M rows): Calendly meeting events. Key columns: `invitee_name` (the `user_xxx` hash), `invitee_email` (real personal email), `starts_at`, `status`.

**`mdl_users_processed`** (1.65M rows): Moodle platform users. Key columns: `username` (unique per account), `firstname` (the `user_xxx` hash — represents the person), `lastname` (also a hash), `lastaccess` (last login timestamp).

**`close_crm_users_processed`** (18.3M rows): CRM user dimension. Key columns: `user_key` (the `user_xxx` hash), `name`, `email`. Used for CSM name resolution.

---
## 5. Gold Layer — Engagement Metrics & Health Profile (5 notebooks, 14 tables)

### 5.1 Gold Notebook Dependency Chain

```
3_1_Gold_Leads_Activities_Summary  (depends on all 11 silver)
    │ → gold.close_crm_users_processed, gold.leads_activities_summary
    ▼
3_2_Gold_Sales_Details  (depends on 3_1)
    │ → gold.sales_details
    ▼
3_3_Gold_Metrics  (depends on 3_2)          3_4_Gold_Lookup_Tables  (depends on 3_1)
    │ → gold.customer_health_base + 7 metrics │ → gold.lead_info, gold.lead_emails
    └──────────────┬───────────────────────────┘
                   ▼
3_5_Gold_Customer_Health_Profile  (depends on 3_3 + 3_4)
    → gold.health_metrics → scoring → gold.customer_health_profile (MERGE SCD1)
```

### 5.2 The 7 Engagement Metric Tables (3_3_Gold_Metrics, cells 3–8)

Each metric is computed as a standalone `CREATE OR REPLACE TABLE` in `crm_ingestion.gold`. All metrics are per-source `lead_id` (no identity consolidation).

| # | Metric Table | Rows | Source | Join Strategy | Business Rule |
|---|---|---|---|---|---|
| 1 | `days_since_last_email` | 5,497 | `lead_activites_processed` + `lead_activites_email_body` | `lead_id` (CRM internal) | TYPE='Email', status='inbox', excludes "A new event has been scheduled." and folder-access requests |
| 2 | `days_since_last_meeting` | 7,110 | `lead_activites_email_body` | `lead_id` (CRM internal) | `body_text ILIKE '%A new event has been scheduled.%'` — CRM-recorded meeting signal |
| 3 | `missed_calls_14_days` | 0 | `lead_activites_processed` | `lead_id` (CRM internal) | TYPE='Call', direction='inbound', disposition='no-answer', last 14 days |
| 4 | `days_until_next_meeting` | 0 | `lead_activites_processed` | `lead_id` (CRM internal) | TYPE='Meeting', status='upcoming', `starts_at > NOW()` |
| 5 | `days_since_last_login` | 356 | `leads_processed_contacts` → `mdl_users_processed` | **Cross-system hash join** (see §6) | `contacts.name = USERNAME ∪ FIRSTNAME ∪ LASTNAME`, returns `DATEDIFF(NOW(), lastaccess)` |
| 6 | `days_since_last_meeting_calendly` | 1,240 | `leads_processed_contacts` → `calendly_scheduled_events_processed` | **Cross-system hash join** | `contacts.name = invitee_name`, returns `DATEDIFF(NOW(), MAX(starts_at))` for past events |
| 7 | `sentiment` | 1,599 | `leads_processed_custom` | `leads_processed_id` (CRM internal) | Reads CRM pre-calculated `SENTIMENTS_LAST_30_DAYS`, `DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT`, `DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER` |

Metrics 3 and 4 have 0 rows — this is a real data timing issue (no missed calls or upcoming meetings at the time of the last pipeline run), not a join bug.

Metric 7 (sentiment) uses CRM's pre-calculated values because a direct join to `student_sentiment_processed` is infeasible — the `channel_id` format in the sentiment table does not match `lead_id` or any CRM identifier. The 13 matches that do exist via `channel_name = contacts.name` are too few to be useful.

### 5.3 Gold Lookup & Supporting Tables

| Table | Rows | Purpose |
|---|---|---|
| `lead_info` | 1,584 | One row per Coaching Client: `lead_id`, `lead_name`, `lead_display_name` |
| `lead_emails` | 43,940 | Array of all email addresses per lead: `lead_id`, `customer_emails` |
| `close_crm_users_processed` | 432 | CRM users filtered to active — used for CSM name resolution |
| `leads_activities_summary` | 800,299 | Aggregated activities per lead (counts by type) |
| `sales_details` | 184 | Sales context: date_of_sale, program, setter_name, closer_name, contract_value |
| `customer_health_base` | 1,584 | Base view: deduplicated Coaching Clients with `lead_id`, `status_label`, `csm_id`, `Contracted_Value`, `Tier`, `Avatar` |

---
## 6. Cross-System Identity Resolution — The Hash Join Strategy

### The Problem

The pipeline integrates data from three independent systems — Close CRM, Calendly, and Moodle — each of which uses different identifiers for the same person:

| System | Identifier | Format |
|---|---|---|
| Close CRM | `leads_processed.id` | `lead_xxx` (CRM-internal, no cross-system use) |
| Close CRM | `contacts.name` | `user_<32-hex MD5 hash>` (e.g., `user_4db1eb7a02944ebefd10ddc14ccb769d`) |
| Calendly | `invitee_name` | Same `user_<32-hex>` format as CRM contacts |
| Moodle | `username` | `user_<32-hex>` but unique per account (26,012 distinct) |
| Moodle | `firstname` | `user_<32-hex>` — represents the PERSON (11,667 distinct, not unique per account) |
| Moodle | `lastname` | `user_<32-hex>` — also not unique (15,324 distinct) |

The `user_xxx` hash is a 128-bit MD5 hash (32 hex characters, prefixed with `user_`) that is **stable and deterministic** — the same person always gets the same hash across systems. This makes it the only viable cross-system join key.

### Why Email Joins Fail

Email addresses appear in all three systems but in incompatible formats:

| System | Email Format |
|---|---|
| CRM | `user_<full 32-hex>@masked.com` (masked) |
| Calendly hosts | Same as CRM (masked) — coaches only, not students |
| Calendly invitees | Real personal emails (gmail.com, yahoo.com) — different string entirely |
| Moodle | `<first 9 hex of username>@masked.com` — truncated, different from CRM |

Email-string joins between CRM and Calendly return **71 matches** (coaches only). Between CRM and Moodle, they return **0 matches**. The hash join is the only path that works.

### Verified Cross-System Match Counts

| Join | Key | Coaching Clients Matched |
|---|---|---|
| CRM `contacts.name` → Calendly `invitee_name` | `user_xxx` | 1,242 |
| CRM `contacts.name` → Moodle `username` only | `user_xxx` | 23 |
| CRM `contacts.name` → Moodle `firstname` only | `user_xxx` | 320 |
| CRM `contacts.name` → Moodle `lastname` only | `user_xxx` | 136 |
| CRM `contacts.name` → Moodle `username ∪ firstname ∪ lastname` | `user_xxx` | **356** |
| CRM `contacts.name` → Calendly `invitee_name` → Moodle `firstname` | `user_xxx` (transitive) | 3,233 (distinct users, not CC-filtered) |

### The Moodle Join Refactor

**Before** (cell 7 of `3_3_Gold_Metrics`): Joined `contacts.name = mdl_users.username` only — matched **23** Coaching Clients. The `username` field is unique per Moodle account, but most CRM contacts don't have a Moodle account with the same hash as `username` — their hash appears in `firstname` or `lastname` instead.

**After**: UNION of three joins, deduplicated by `lead_id`, prioritizing most recent `lastaccess`:

```sql
WITH matches AS (
  SELECT cc.lead_id, m.lastaccess
  FROM coaching_clients cc
  JOIN contacts c ON cc.id = c.leads_processed_id
  JOIN mdl_users m ON c.name = m.username  -- 23 matches
  UNION
  SELECT cc.lead_id, m.lastaccess
  FROM coaching_clients cc
  JOIN contacts c ON cc.id = c.leads_processed_id
  JOIN mdl_users m ON c.name = m.firstname  -- 320 matches
  UNION
  SELECT cc.lead_id, m.lastaccess
  FROM coaching_clients cc
  JOIN contacts c ON cc.id = c.leads_processed_id
  JOIN mdl_users m ON c.name = m.lastname   -- 136 matches
)
SELECT lead_id, MIN(DATEDIFF(CURRENT_DATE, lastaccess)) AS days_since_last_login
FROM (
  SELECT lead_id, lastaccess, ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY lastaccess DESC) AS rn
  FROM matches
) WHERE rn = 1
GROUP BY lead_id
```

**Result: 23 → 356 Coaching Clients (15× improvement).**

The Calendly join (`contacts.name = invitee_name`) was already hash-based and needed no change — it already matches 1,242 of 1,584 clients (78.3%).

### What Was Removed: `record_hash`

A previous version of `write_silver()` added an `add_record_hash()` call that computed `MD5(concat_ws('|', all columns))` as a row fingerprint. This was removed because:
- It cannot be used for joins (it's per-row, not per-entity)
- It cannot be used for cross-system identity resolution (it hashes all columns, not the identity key)
- CDC is not implemented (no watermark-based deduplication needed)
- It added write overhead without any downstream consumer

The `user_xxx` hash already present in `contacts.name` is the true identity key — it is stable, deterministic, and present across all three systems.

---
## 7. Health Score Logic — Flags → Patterns → Scores → Bands

The scoring pipeline in `3_5_Gold_Customer_Health_Profile` (cell 2) uses a 4-stage CTE chain. All thresholds are hardcoded business rules (see §7.4).

### 7.1 Stage 1: Binary Flags

Three engagement dimensions, each producing a binary flag (1 = engaged, 0 = not engaged, NULL = no data):

| Flag | Metric(s) | Threshold | Logic |
|---|---|---|---|
| `slack_engaged_flag` | `days_since_last_message_from_client` | ≤14 days | NULL if no sentiment data; 1 if ≤14d; 0 if >14d |
| `platform_engaged_flag` | `days_since_last_login` | ≤14 days | NULL if no Moodle match; 1 if ≤14d; 0 if >14d |
| `meeting_engaged_flag` | `days_since_last_meeting` OR `days_since_last_meeting_calendly` | ≤30 days | NULL only when BOTH sources are NULL; 1 if either ≤30d; 0 if both >30d |

**Critical fix:** The meeting flag originally used `OR` for the NULL check, which meant Calendly NULL killed CRM data. Changed to `AND` so the flag is NULL only when BOTH sources are NULL.

### 7.2 Stage 2: Engagement Pattern

The three flags combine into 9 ordered patterns:

| Pattern | Slack | Platform | Meeting | Base Score |
|---|---|---|---|---|
| SLACK + PLATFORM + MEETING | 1 | 1 | 1 | 90 |
| SLACK + PLATFORM | 1 | 1 | 0 or NULL | 80 |
| PLATFORM + MEETING | 0 or NULL | 1 | 1 | 75 |
| SLACK + MEETING | 1 | 0 or NULL | 1 | 60 |
| PLATFORM ONLY | 0 or NULL | 1 | 0 or NULL | 50 |
| MEETING ONLY | 0 or NULL | 0 or NULL | 1 | 40 |
| SLACK ONLY | 1 | 0 or NULL | 0 or NULL | 30 |
| NO ENGAGEMENT | 0 | 0 | 0 or NULL | 20 |
| NO DATA | (insufficient data) | | | NULL |

### 7.3 Stage 3: Adjustments and Final Score

Two adjustments modify the base score:

| Adjustment | Condition | Value |
|---|---|---|
| `missed_call_adjustment` | 0 missed calls | +5 |
| | 1–2 missed calls | 0 |
| | ≥3 missed calls | −15 |
| `sentiment_adjustment` | POSITIVE | +10 |
| | NEUTRAL | +5 |
| | NEGATIVE | −25 |
| | No sentiment | 0 |

**Final health score** = `base_score + missed_call_adjustment + sentiment_adjustment` (NULL if pattern is NO DATA).

### 7.4 Stage 4: Health Band

| Band | Score Range | Current Count | % |
|---|---|---|---|
| GOOD | ≥75 | 190 | 12.0% |
| AVERAGE | 50–74 | 303 | 19.1% |
| POOR | <50 | 476 | 30.1% |
| NO DATA | NULL | 615 | 38.8% |

### Hardcoded Business Rules (not alterable without approval)

| Rule | Value |
|---|---|
| CRM email filter | TYPE='Email' AND status='inbox' |
| Email exclusion | "A new event has been scheduled." (excluded from email metric, used as meeting signal) |
| Email exclusion | "is requesting access to the following folder" |
| Email exclusion | "requests access to an item" |
| CRM meeting signal | "A new event has been scheduled." in `body_text` |
| Missed call lookback | 14 days |
| Missed call criteria | TYPE='Call', direction='inbound', disposition='no-answer' (NOT 'busy') |
| Communication engagement threshold | 14 days |
| Platform engagement threshold | 14 days |
| Meeting engagement threshold | 30 days |
| Excluded Slack channel | `C082U471YMU` |

---
## 8. Work Performed & Impact

### 8.1 Bugs Fixed During the Build

| # | Bug | Root Cause | Fix | Impact |
|---|---|---|---|---|
| 1 | Silver `write_silver` schema merge failure | `mergeSchema: true` caused `DELTA_FAILED_TO_MERGE_FIELDS` on schema changes | Changed to `overwriteSchema: true` | All silver tables can be re-run without schema conflicts |
| 2 | `extract_custom` explode failure | `explode_outer().cast("string")` nesting caused `UNSUPPORTED_GENERATOR.NESTED_IN_EXPRESSIONS` | Explode first, cast second | Custom field extraction works on all tables |
| 3 | Child table PK naming collision | `explode_to_child` only renamed parent PK on name collision | Always rename parent PK to `{parent_name}_id` via `parent_pk` parameter | FK columns are unambiguous across all child tables (verified 3 levels deep) |
| 4 | Gold email metric using wrong column | Email text exclusions checked `activity_text` (NULL for all email+inbox rows) | Changed to `body_preview` with `coalesce(..., lit(False))` | Email exclusion filters now work correctly |
| 5 | Gold CRM meeting using wrong column | "A new event has been scheduled" searched in `activity_text` (NULL) | Changed to `body_text` from `lead_activites_email_body` | 13,519 meeting signals detected (was 0) |
| 6 | Gold meeting flag NULL logic | `meeting_engaged_flag` used `|` (OR) so Calendly NULL killed CRM data | Changed to `&` (AND) so flag is NULL only when BOTH sources NULL | Clients with CRM meeting data but no Calendly data are now scored |
| 7 | Gold mart row multiplication | Multiple emails per lead caused 1,846 rows vs 1,469 unique leads | Added email dedup to one-per-lead | Health profile has exactly 1,584 rows (one per Coaching Client) |
| 8 | Bronze `parse_bronze_table` all-NULL fields | When UDF returns JSON array string, `from_json()` with `StructType` produces all NULLs | Track `_udf_returns_array` flag, use `ArrayType(deduped_st)` when true | Fixed `close_crm_users_raw` (18.3M rows) and `lead_activites_raw` (9.2M rows) |
| 9 | Moodle login join coverage | `contacts.name = mdl_users.username` matched only 23 clients | UNION of `username` + `firstname` + `lastname`, dedup by `lead_id` | **23 → 356 clients (15× improvement)** |
| 10 | `record_hash` overhead | MD5 all-columns fingerprint had no use case (can't join, can't resolve identity, no CDC) | Removed `add_record_hash()` from `write_silver()` | Eliminated unnecessary write overhead on every silver table |

### 8.2 Quantified Impact

**Before the Moodle join refactor:**

| Metric | Value |
|---|---|
| Clients with `days_since_last_login` | 23 (1.5%) |
| Clients with `platform_engaged_flag` = 0 or 1 | 23 |
| Clients with `platform_engaged_flag` = NULL | 1,561 (98.5%) |
| Clients in NO DATA band | ~948 |
| Clients with a real health score | ~636 |

**After the Moodle join refactor:**

| Metric | Value |
|---|---|
| Clients with `days_since_last_login` | 356 (22.5%) |
| Clients with `platform_engaged_flag` = 0 (not engaged) | 356 |
| Clients with `platform_engaged_flag` = NULL | 1,228 (77.5%) |
| Clients in NO DATA band | 615 (38.8%) |
| Clients with a real health score | 969 (61.2%) |

**Net impact: 333 clients moved from unscoreable ("NO DATA") to scored health bands.**

### 8.3 Pipeline Data Volume Summary

| Layer | Tables | Total Rows | Largest Table |
|---|---|---|---|
| Bronze | 9 | 17,246,500 | `calendly_scheduled_events` (13.5M) |
| Silver | 11 | 44,223,263 | `close_crm_users_processed` (18.3M) |
| Gold | 14 | 853,589 | `leads_activities_summary` (800K) |
| **Total** | **34** | **62,323,352** | |

---
## 9. Profile Evaluation — Before vs After (Live Data)

The following cells query the live `customer_health_profile` table to demonstrate the pipeline's actual output.

In [0]:
%sql
-- Engagement pattern distribution
SELECT 
    engagement_pattern,
    COUNT(*) AS leads,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM crm_ingestion.gold.customer_health_profile
GROUP BY engagement_pattern
ORDER BY leads DESC

In [0]:
%sql
-- Health band distribution with score stats
SELECT 
    health_band,
    COUNT(*) AS leads,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(final_health_score), 1) AS avg_score,
    MIN(final_health_score) AS min_score,
    MAX(final_health_score) AS max_score
FROM crm_ingestion.gold.customer_health_profile
GROUP BY health_band
ORDER BY 
    CASE health_band 
        WHEN 'GOOD' THEN 1 
        WHEN 'AVERAGE' THEN 2 
        WHEN 'POOR' THEN 3 
        ELSE 4 
    END

In [0]:
%sql
-- Metric coverage in health_metrics (how many leads have each metric populated)
SELECT
    COUNT(*) AS total_leads,
    COUNT(days_since_last_email) AS email,
    ROUND(COUNT(days_since_last_email)*100.0/COUNT(*), 1) AS email_pct,
    COUNT(days_since_last_meeting) AS crm_meeting,
    ROUND(COUNT(days_since_last_meeting)*100.0/COUNT(*), 1) AS crm_meeting_pct,
    COUNT(days_since_last_meeting_calendly) AS calendly,
    ROUND(COUNT(days_since_last_meeting_calendly)*100.0/COUNT(*), 1) AS calendly_pct,
    COUNT(days_since_last_login) AS platform_login,
    ROUND(COUNT(days_since_last_login)*100.0/COUNT(*), 1) AS login_pct,
    COUNT(sentiment) AS sentiment,
    ROUND(COUNT(sentiment)*100.0/COUNT(*), 1) AS sentiment_pct
FROM crm_ingestion.gold.health_metrics

In [0]:
%sql
-- Sample GOOD profiles (score >= 75)
SELECT
    lead_name,
    engagement_pattern,
    base_score,
    missed_call_adjustment AS missed_adj,
    sentiment_adjustment AS sentiment_adj,
    final_health_score,
    health_band,
    calc_days_since_last_email AS email_days,
    calc_days_since_last_login AS login_days,
    calc_days_since_last_meeting_calendly AS calendly_days,
    sentiment AS sentiment_label
FROM crm_ingestion.gold.customer_health_profile
WHERE health_band = 'GOOD'
ORDER BY final_health_score DESC
LIMIT 5

In [0]:
%sql
-- Sample POOR profiles (score < 50)
SELECT
    lead_name,
    engagement_pattern,
    base_score,
    missed_call_adjustment AS missed_adj,
    sentiment_adjustment AS sentiment_adj,
    final_health_score,
    health_band,
    calc_days_since_last_email AS email_days,
    calc_days_since_last_login AS login_days,
    calc_days_since_last_meeting_calendly AS calendly_days,
    sentiment AS sentiment_label
FROM crm_ingestion.gold.customer_health_profile
WHERE health_band = 'POOR'
ORDER BY final_health_score ASC
LIMIT 5

In [0]:
%sql
-- Platform engaged flag: the metric that changed from the Moodle join refactor
SELECT 
    CASE 
        WHEN platform_engaged_flag IS NULL THEN 'NULL — no Moodle match (unscoreable)'
        WHEN platform_engaged_flag = 1 THEN 'ENGAGED — last login <=14 days'
        WHEN platform_engaged_flag = 0 THEN 'NOT ENGAGED — last login >14 days'
    END AS platform_status,
    COUNT(*) AS leads,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM crm_ingestion.gold.customer_health_profile
GROUP BY 1
ORDER BY 1

---
## 10. Profile Evaluation — Before vs After

### How a profile was evaluated BEFORE the Moodle join fix

A coaching client with a Moodle account whose hash appeared in `firstname` (not `username`) would have:
- `days_since_last_login` = NULL (no match via `username`)
- `platform_engaged_flag` = NULL
- Engagement pattern = 'NO DATA' (if other metrics also NULL) or 'SLACK + MEETING' (if slack and meeting data existed but platform was NULL)
- Health score = NULL (unscoreable)
- Health band = 'NO DATA'

This client would appear in dashboards as a blank — no score, no band, no actionable insight.

### How the same profile is evaluated AFTER the fix

The same client now has:
- `days_since_last_login` = a real number (e.g., 20 days)
- `platform_engaged_flag` = 0 (not engaged, last login >14 days)
- Engagement pattern = 'NO ENGAGEMENT' (if all three flags are 0) or 'SLACK ONLY' / 'MEETING ONLY' (if other engagement exists)
- Health score = a real number (e.g., 20 + adjustments)
- Health band = 'POOR' or 'AVERAGE'

The client is now **actionable** — they appear in the POOR or AVERAGE band and can be prioritized for outreach.

### Concrete Example

| Field | Before (USERNAME only) | After (UNION) |
|---|---|---|
| `days_since_last_login` | NULL | 20 |
| `platform_engaged_flag` | NULL | 0 |
| `engagement_pattern` | NO DATA | NO ENGAGEMENT |
| `base_score` | NULL | 20 |
| `final_health_score` | NULL | 25 (20 + 5 missed_call + 0 sentiment) |
| `health_band` | NO DATA | POOR |

### Pipeline Value

**Before this pipeline existed:** Close CRM had pre-calculated custom fields (`DAYS_SINCE_LAST_LOGIN`, `FINAL_SCORE`, `HEALTH_BAND`) that were opaque — no one could verify how they were computed, what data fed them, or whether they were accurate.

**After this pipeline:**
1. **Transparent scoring**: Every metric is computed from raw activity data with visible SQL. The scoring logic (flags → patterns → scores → bands) is documented and auditable.
2. **Cross-system coverage**: The pipeline joins CRM, Calendly, and Moodle data — something Close CRM's internal calculations cannot do. 1,240 clients now have Calendly meeting data; 356 have Moodle login data.
3. **Actionable segmentation**: 969 clients (61.2%) now have a health score. 476 are in POOR (need outreach), 303 are in AVERAGE (monitor), 190 are in GOOD (sustain). The 615 in NO DATA need additional data sources.
4. **Validated against CRM**: The pipeline's self-calculated metrics can be compared against Close CRM's pre-calculated `leads_processed_custom` values to identify discrepancies.
5. **Daily automated refresh**: The Databricks Job runs 17 tasks daily at 8:00 AM EST, so scores are always current.

### What's Next

| Opportunity | Current | Potential |
|---|---|---|
| Moodle login coverage | 356 (22.5%) | Higher — `firstname` and `lastname` are not unique per account; a person-level grouping could increase coverage |
| Sentiment cross-system join | 0 (infeasible via `channel_id`) | If `channel_name` matches `contacts.name`, 13 matches exist — too few for production but worth monitoring |
| Missed calls (14-day window) | 0 rows | Data timing — no missed calls at last run; will populate when data arrives |
| Upcoming meetings | 0 rows | Data timing — no future meetings at last run; will populate when meetings are scheduled |
| CDC (incremental processing) | Not implemented | `write_silver()` has CDC scaffolding (watermark check, delete+insert); enabling it would avoid full overwrites |
| `record_hash` column cleanup | Still in old silver tables | Re-running silver notebooks will remove it (no longer added by `write_silver()`) |